# HAIKUTHON 참가자 시뮬레이션 (Colab L4, 24GB CUDA)

**과제**: 텍스트 프롬프트 8,000개마다 15,000장(정답 8,000 + hard negative 7,000) 중 정답 이미지를 순위화.
점수 = (Recall@1 + NDCG@5) / 2. 문항별 독립 예측, 24GB GPU.

**이 노트북이 하는 일** (공용 코드 `baselines/chan_lee` 재사용, 결과는 `baselines/seohyun_jang/`에 저장)
1. B1: SigLIP2 zero-shot — CUDA 시간·VRAM 실측
2. B2: SigLIP2 파인튜닝(묶음 단위 hard in-batch negative) — train으로 학습, val로 확인
3. B3: B2 top-K를 Qwen3.5-4B yes/no로 재정렬 — **test 전체 8,000문항** (chan_lee는 400문항 부분집합만 측정)
4. P: 익명화한 blind test 패키지로 채점 서버처럼 처음부터 끝까지 시간 측정 → 제출 CSV 채점

**기밀 규칙**: test 프롬프트·이미지·정답을 출력하지 않습니다. 셀 출력은 지표와 로그만 나오게 되어 있습니다.
이 노트북에서는 Gemini 코드 도우미를 쓰지 마세요.

**준비**: 런타임 → 런타임 유형 변경 → **L4 GPU**. 왼쪽 🔑 Secrets에 `GH_TOKEN`(classic, repo 권한) 추가, 노트북 액세스 켜기.
Drive 공유 폴더의 `train_v1.tar`, `test_v1.tar`를 **내 드라이브에 바로가기 추가**.

## 0. GPU 확인

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import sys; print(sys.version)

## 1. Drive 마운트 · 레포 클론 · 패키지 설치 (설치 후 런타임이 자동 재시작됩니다)

In [ ]:
import os, subprocess
from google.colab import drive, userdata
drive.mount('/content/drive')

REPO = '/content/haikuthon'
tok = userdata.get('GH_TOKEN')
url = f'https://{tok}@github.com/3sirn3203/26-haikuthon.git'
if not os.path.exists(REPO):
    r = subprocess.run(['git', 'clone', '-q', url, REPO], capture_output=True, text=True)
else:
    r = subprocess.run(['git', '-C', REPO, 'pull', '-q', '--rebase', 'origin', 'main'], capture_output=True, text=True)
print('git ok' if r.returncode == 0 else r.stderr.replace(tok, '***'))

# Colab의 CUDA torch를 그대로 쓰고 나머지만 설치
req = open(f'{REPO}/baselines/chan_lee/requirements.txt').read().splitlines()
keep = [l for l in req if l.strip() and not l.startswith('#') and not l.split('==')[0] in ('torch', 'torchvision')]
open('/content/req_colab.txt', 'w').write('\n'.join(keep) + '\n')
!pip install -q -r /content/req_colab.txt
print('설치 완료 -> 런타임 재시작')
os.kill(os.getpid(), 9)

## 2. 환경 변수 (재시작 후 · 세션이 끊긴 뒤에는 여기부터 다시 실행)

In [ ]:
import os, json, glob, shutil, subprocess
from pathlib import Path
from google.colab import drive
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')

REPO = Path('/content/haikuthon')
CHAN = REPO / 'baselines/chan_lee'
ME = REPO / 'baselines/seohyun_jang'
DATA = REPO / 'data'
CACHE = CHAN / 'cache'                      # 로컬 디스크 (chan_lee/.gitignore로 제외됨)
BACKUP = Path('/content/drive/MyDrive/haiku_backup')   # 체크포인트·재정렬 점수 백업
os.environ.update(HAIKU_DATA_ROOT=str(DATA), HAIKU_CACHE_DIR=str(CACHE), PYTHONDONTWRITEBYTECODE='1',
                  TOKENIZERS_PARALLELISM='false')
os.chdir(CHAN)

import torch, transformers, open_clip
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
print('transformers', transformers.__version__, '| open_clip', open_clip.__version__)

def show(name):
    """결과 요약만 출력 (프롬프트·정답 없음)."""
    m = json.loads((CHAN / 'results' / name / 'metrics.json').read_text())
    s, by, t = m['scores'], m.get('scores_by_split', {}), m.get('timing', {})
    print(f"{name}: score={s['score']:.4f} R@1={s['recall_at_1']:.4f} NDCG@5={s['ndcg_at_5']:.4f} "
          + ' '.join(f"{k}={v['score']:.4f}" for k, v in by.items())
          + f" | n={m.get('n_queries')} total={t.get('total_sec', 0):.0f}s device={t.get('device')} "
          f"peak={m.get('peak_vram_gb', 0):.2f}GB")

def backup():
    BACKUP.mkdir(parents=True, exist_ok=True)
    for sub in ('ckpt', 'rerank', 'meta'):
        if (CACHE / sub).exists():
            shutil.copytree(CACHE / sub, BACKUP / sub, dirs_exist_ok=True)
    print('backup ->', BACKUP)

def restore():
    for sub in ('ckpt', 'rerank', 'meta'):
        if (BACKUP / sub).exists():
            shutil.copytree(BACKUP / sub, CACHE / sub, dirs_exist_ok=True)
restore()

## 3. 데이터 풀기 (처음 한 번, test 13GB라 10~20분)

In [ ]:
def find_tar(name):
    for p in [f'/content/drive/MyDrive/{name}', f'/content/drive/MyDrive/haikuthon/{name}']:
        if os.path.exists(p):
            return p
    hits = glob.glob(f'/content/drive/MyDrive/*/{name}') + glob.glob(f'/content/drive/MyDrive/*/*/{name}')
    if not hits:
        raise FileNotFoundError(f'{name}: 공유 폴더에서 내 드라이브로 바로가기를 추가하세요')
    return hits[0]

DATA.mkdir(exist_ok=True)
(DATA / '.gitignore').write_text('*\n')
for split in ('train_v1', 'test_v1'):
    if (DATA / split / 'prompts.jsonl').exists():
        print(split, '이미 있음'); continue
    tar = find_tar(f'{split}.tar')
    # 절대경로/상위경로 항목이 없는지 먼저 검사
    names = subprocess.run(['tar', '-tf', tar], capture_output=True, text=True, check=True).stdout.splitlines()
    bad = [n for n in names if n.startswith('/') or '..' in n.split('/')]
    assert not bad, f'unsafe entries: {len(bad)}'
    subprocess.run(['tar', '-xf', tar, '-C', str(DATA), '--no-same-owner'], check=True)
    print(split, 'done')
for split in ('train_v1', 'test_v1'):
    n = sum(1 for _ in (DATA / split / 'images').rglob('*.png'))
    print(split, 'images:', n)   # train 4500, test 15000 기대

## 4. 공용 준비: HN 짝 정보(Public/Private 분할) · 동작 확인

In [ ]:
if not (CACHE / 'meta' / 'test_hn_sources.json').exists():
    !python scripts/pair_hn.py --split train > /dev/null && python scripts/pair_hn.py --split test > /dev/null
    backup()
!python scripts/run_exp.py --config configs/exp/b0_random_test.yaml 2>&1 | tail -1

## 5. 내 실험 설정 (exp_id = seohyun_jang/...)

In [ ]:
CK = CACHE / 'ckpt' / 'b2_siglip2_bundle_l4.pt'
SIG = "{type: openclip, model: ViT-SO400M-16-SigLIP2-384, pretrained: webli, precision: fp16, batch_size: 64, num_workers: 8"
QWEN = "{type: vlm_yesno, model: Qwen/Qwen3.5-4B, dtype: bf16, batch_size: 10, max_pixels: 200704"
cfgs = {
 'b1_siglip2_so400m384_l4': f'''exp_id: seohyun_jang/b1_siglip2_so400m384_l4
category: honest
method: "B1 zero-shot SigLIP2 so400m/16@384, full pool cosine; CUDA L4 timing (no cache)"
eval_split: test
retriever: {SIG}}}
''',
 'b2_siglip2_bundle_l4': f'''exp_id: seohyun_jang/b2_siglip2_bundle_l4
category: honest
method: "B2 fine-tuned SigLIP2 so400m (last 4 blocks + heads, sigmoid loss), whole stage1 bundles per batch; trained on CUDA L4"
uses_train_finetune: true
uses_train_hard_negatives: false
eval_split: test
retriever: {SIG}}}
finetune: {{epochs: 5, lr: 1.0e-5, batch_size: 32, train_last_blocks: 4, batching: bundle, use_hard_negatives: false, num_workers: 8, seed: 0}}
''',
 'b2_siglip2_bundle_l4_stage1': f'''exp_id: seohyun_jang/b2_siglip2_bundle_l4
category: honest
method: "stage-1 of b2_siglip2_bundle_l4 (fine-tuned checkpoint), base for B3"
uses_train_finetune: true
eval_split: test
retriever: {SIG}, checkpoint: {CK}}}
''',
 'b3_siglip2_bundle_l4__qwen35_4b_full': f'''exp_id: seohyun_jang/b3_siglip2_bundle_l4__qwen35_4b_full
category: honest
method: "B3 two-stage: fine-tuned SigLIP2 (b2_siglip2_bundle_l4) top-K reranked by Qwen3.5-4B yes/no logit; FULL test"
stage1_config: configs/exp/sj_b2_siglip2_bundle_l4_stage1.yaml
reranker: {QWEN}}}
k_max: 10
k_list: [3, 5, 10]
subset: {{private: 7000, public: 1000, seed: 0}}
''',
 'b4_e2e_siglip2_bundle_l4__qwen35_4b_k5': f'''exp_id: seohyun_jang/b4_e2e_siglip2_bundle_l4__qwen35_4b_k5
category: honest
method: "end-to-end blind inference (predict.py): fine-tuned SigLIP2 top-5 -> Qwen3.5-4B rerank"
uses_train_finetune: true
eval_split: test
retriever: {SIG}, checkpoint: {CK}}}
reranker: {QWEN}, top_k: 5}}
''',
}
for k, v in cfgs.items():
    (CHAN / 'configs/exp' / f'sj_{k}.yaml').write_text(v)
print('\n'.join(sorted(p.name for p in (CHAN / 'configs/exp').glob('sj_*'))))

## 6. B1 zero-shot (CUDA 시간 실측, 약 10~15분)

In [ ]:
!python scripts/run_exp.py --config configs/exp/sj_b1_siglip2_so400m384_l4.yaml --no-cache 2>&1 | grep -v -i warn | tail -3
show('b1_siglip2_so400m384_l4')

## 7. B2 파인튜닝 → val · test 평가 (약 30~60분)

In [ ]:
!python scripts/finetune.py --config configs/exp/sj_b2_siglip2_bundle_l4.yaml 2>&1 | grep -E '^(train pairs|ep .* step [0-9]+0/|seohyun)|train_sec|train_peak' | tail -25
backup()
show('b2_siglip2_bundle_l4__val'); show('b2_siglip2_bundle_l4')

## 8. B3 재정렬 — test 전체 8,000문항 × top-10 (중단돼도 이어서 실행됨, 백업 후 재실행)

In [ ]:
!python scripts/run_rerank.py --config configs/exp/sj_b3_siglip2_bundle_l4__qwen35_4b_full.yaml 2>&1 | grep -E 'queries|budget|K=|seohyun|Error' | tail -15
backup()
for K in (3, 5, 10):
    show(f'b3_siglip2_bundle_l4__qwen35_4b_full_k{K}')

## 9. 채점 서버처럼: 익명화 test 패키지 → predict.py (시간 포함) → 채점

In [ ]:
if not (CACHE / 'blind/test/prompts.jsonl').exists():
    !python scripts/make_blind_test.py > /dev/null
!python scripts/predict.py --config configs/exp/sj_b4_e2e_siglip2_bundle_l4__qwen35_4b_k5.yaml \
    --images cache/blind/test/images --prompts cache/blind/test/prompts.jsonl \
    --out cache/sub_b4.csv --timing-out cache/timing_b4.json > /dev/null 2>&1
t = json.loads((CACHE / 'timing_b4.json').read_text())
print({k: t[k] for k in ('device', 'total_sec_including_startup', 'peak_vram_gb', 'n_queries', 'n_images') if k in t})
sc = subprocess.run(['python', 'scripts/make_blind_test.py', '--score', 'cache/sub_b4.csv'], capture_output=True, text=True).stdout
print(sc)
(CACHE / 'score_b4.json').write_text(sc)

## 10. 결과를 내 폴더로 복사 · 커밋 · 푸시 (본인 폴더만)

In [ ]:
mine = [p.name for p in (CHAN / 'results').iterdir()
        if p.is_dir() and (p / 'metrics.json').exists()
        and json.loads((p / 'metrics.json').read_text()).get('author') == 'seohyun_jang']
for n in mine:
    shutil.copytree(CHAN / 'results' / n, ME / 'results' / n, dirs_exist_ok=True)
    shutil.rmtree(CHAN / 'results' / n)            # chan_lee 폴더에 흔적 남기지 않음
for p in (CHAN / 'configs/exp').glob('sj_*.yaml'):
    shutil.copy(p, ME / 'configs' / p.name) if (ME / 'configs').exists() else ((ME / 'configs').mkdir(), shutil.copy(p, ME / 'configs' / p.name))
# 9번 blind 채점 결과는 metrics.json 형식으로 따로 저장
if (CACHE / 'score_b4.json').exists():
    d = ME / 'results/b4_e2e_siglip2_bundle_l4__qwen35_4b_k5'; d.mkdir(parents=True, exist_ok=True)
    sc = json.loads((CACHE / 'score_b4.json').read_text()); t = json.loads((CACHE / 'timing_b4.json').read_text())
    a = sc['all']
    (d / 'metrics.json').write_text(json.dumps({
        'exp_id': 'seohyun_jang/b4_e2e_siglip2_bundle_l4__qwen35_4b_k5', 'author': 'seohyun_jang', 'category': 'honest',
        'method': 'end-to-end blind inference via predict.py (no cache, timed from process start)',
        'eval_split': 'test_full', 'n_queries': a['n'], 'pool_size': t.get('n_images'),
        'stage1_model': 'ViT-SO400M-16-SigLIP2-384:webli (fine-tuned)', 'stage2_model': 'Qwen/Qwen3.5-4B', 'top_k': 5,
        'uses_train_finetune': True, 'uses_train_hard_negatives': False, 'violates_cross_query_rule': False,
        'scores': a, 'scores_by_split': {k: v for k, v in sc.items() if k != 'all'},
        'timing': {'device': t.get('device'), 'total_sec': t.get('total_sec_including_startup'), **{k: v for k, v in t.items() if k.endswith('_sec')}},
        'peak_vram_gb': t.get('peak_vram_gb'), 'params_b': t.get('params_b')}, indent=2))
    shutil.copy(CHAN / 'configs/exp/sj_b4_e2e_siglip2_bundle_l4__qwen35_4b_k5.yaml', d / 'config.yaml')
    if not (d / 'notes.md').exists():
        (d / 'notes.md').write_text('# b4_e2e\n\n채점 서버 형식 end-to-end 측정 (Colab L4).\n\n## 관찰\n\n- \n')
print(sorted(p.name for p in (ME / 'results').iterdir()))

In [ ]:
GIT_NAME = ''    # 예: 'Seohyun Jang'
GIT_EMAIL = ''   # GitHub 계정 이메일 또는 noreply 주소
assert GIT_NAME and GIT_EMAIL, '이름과 이메일을 채우세요'
os.chdir(REPO)
!git config user.name "{GIT_NAME}" && git config user.email "{GIT_EMAIL}"
!git pull -q --rebase origin main
!git add baselines/seohyun_jang
!git status --short | grep -v '^?? ' | head -40
!git diff --cached --stat | tail -3

In [ ]:
# 위 목록에 baselines/seohyun_jang/ 밖의 파일이나 데이터·가중치·CSV가 없을 때만 실행
!git commit -q -m "seohyun_jang: Colab L4 baselines (B1, B2, B3 full test, end-to-end)" && git push -q origin main && git log --oneline -1
os.chdir(CHAN)